# Context-FID Score Presentation
## Necessary packages and functions call

- Context-FID score: A useful metric measures how well the the synthetic time series windows ”fit” into the local context of the time series

In [1]:
import os
import torch
import numpy as np
import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))
from Utils.context_fid import Context_FID
from Utils.metric_utils import display_scores
from Utils.cross_correlation import CrossCorrelLoss

## Data Loading

Load original dataset and preprocess the loaded data.

In [4]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
ori_data = np.load('../OUTPUT/test/samples/evening_peak_etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/test/ddpm_fake_evening_peak_etth_milestone_500_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(0, 2, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2880, 24, 7)
fake shape is:  (20384, 24, 1)
ori shape is:  (20160, 24, 1)
fake shape is:  (20384, 24, 1)


## Context-FID Score

- The Frechet Inception distance-like score is based on unsupervised time series embeddings. It is able to score the fit of the fixed length synthetic samples into their context of (often much longer) true time series.

- The lowest scoring models correspond to the best performing models in downstream tasks

In [5]:
for j in range(10):

    context_fid_score = []

    for i in range(iterations):
        context_fid = Context_FID(ori_data[:], fake_data[:ori_data.shape[0]])
        context_fid_score.append(context_fid)
        print(f'Iter {i}: ', 'context-fid =', context_fid, '\n')

    display_scores(context_fid_score)

# Seed 12345 Final Score:  0.13663267402399055 ± 0.005536055372540525
# Seed 12345 Final Score:  0.129
# Seed 12345 Final Score:  0.135

# Seed 2025 Final Score:  0.148
# Seed 2025 Final Score:  0.147
# Seed 2025 Final Score:  0.142
# Seed 2025 Final Score:  0.148

# univariate single ep,  500 epoch kankan:   0.009

Iter 0:  context-fid = 0.009376299163037826 

Iter 1:  context-fid = 0.009216126083815635 

Iter 2:  context-fid = 0.008509468968000476 

Iter 3:  context-fid = 0.00909709890316055 

Iter 4:  context-fid = 0.008778091039053188 

Final Score:  0.008995416831413534 ± 0.0004334719060097304
Iter 0:  context-fid = 0.008848024288531639 

Iter 1:  context-fid = 0.007710494256759121 

Iter 2:  context-fid = 0.009267999673275882 

Iter 3:  context-fid = 0.008538325410980421 

Iter 4:  context-fid = 0.010013355119903996 

Final Score:  0.008875639749890212 ± 0.0010609599585429499
Iter 0:  context-fid = 0.008862112979035925 

Iter 1:  context-fid = 0.008392777432354719 

Iter 2:  context-fid = 0.008465749593588662 

Iter 3:  context-fid = 0.008454290750425886 

Iter 4:  context-fid = 0.008775262001526288 

Final Score:  0.008590038551386297 ± 0.00026421444354871563
Iter 0:  context-fid = 0.008774442961688389 

Iter 1:  context-fid = 0.008053495142619438 

Iter 2:  context-fid = 0.0091086018437401

## Correlational Score

- The metric uses the absolute error of the auto-correlation estimator by real data and synthetic data as the metric to assess the temporal dependency.

- For d > 1, it uses the l1-norm of the difference between cross correlation matrices.

In [ ]:
def random_choice(size, num_select=100):
    select_idx = np.random.randint(low=0, high=size, size=(num_select,))
    return select_idx

In [ ]:
x_real = torch.from_numpy(ori_data)
x_fake = torch.from_numpy(fake_data)

correlational_score = []
size = int(x_real.shape[0] / iterations)

for i in range(iterations):
    real_idx = random_choice(x_real.shape[0], size)
    fake_idx = random_choice(x_fake.shape[0], size)
    corr = CrossCorrelLoss(x_real[real_idx, :, :], name='CrossCorrelLoss')
    loss = corr.compute(x_fake[fake_idx, :, :])
    correlational_score.append(loss.item())
    print(f'Iter {i}: ', 'cross-correlation =', loss.item(), '\n')

display_scores(correlational_score)